# 04 · Placar das três configurações, alucinações e erros
Lê as respostas salvas pelos notebooks 02 e 03 (não precisa de GPU).

In [ ]:
# Preparação do ambiente (Colab ou local)
import os, sys, subprocess
REPO_URL = "https://github.com/giuliano10/GS-GENENERATIVE-AI.git"
SUBPASTA = "assistente-acessibilidade-calcadas"   # pasta do projeto dentro do repositório

if "google.colab" in sys.modules:
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", REPO_URL, "/content/repo"], check=True)
    else:   # mesma sessão de outro notebook: atualiza o código
        subprocess.run(["git", "-C", "/content/repo", "pull", "-q"])
    os.chdir(os.path.join("/content/repo", SUBPASTA))
    # O Colab já traz torch, transformers, numpy etc. Instalamos SÓ o que falta, sem fixar
    # versões: reinstalar o torch com a sessão aberta quebra o "import torch".
    import importlib.util
    faltando = [pip for mod, pip in [("peft", "peft"), ("accelerate", "accelerate"),
                ("sentence_transformers", "sentence-transformers"), ("fastapi", "fastapi"),
                ("httpx", "httpx"), ("tabulate", "tabulate"), ("bs4", "beautifulsoup4"),
                ("lxml", "lxml")] if importlib.util.find_spec(mod) is None]
    # O torchao que vem no Colab é antigo e faz o peft recusar o import; o projeto não o usa.
    if importlib.util.find_spec("torchao") is not None:
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
    if faltando:
        print("Instalando:", faltando)
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltando],
                           capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stdout[-3000:], r.stderr[-3000:])
            raise RuntimeError("Falha no pip install (detalhes acima)")
    # Cada notebook do Colab roda numa máquina separada: os resultados passam de um notebook
    # para o outro por uma pasta no Google Drive.
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = "/content/drive/MyDrive/assistente-calcadas-resultados"
    os.makedirs(PERSIST, exist_ok=True)
    subprocess.run(["cp", "-r", f"{PERSIST}/.", "."])
    print("Resultados anteriores trazidos do Drive:", sorted(os.listdir(PERSIST)))
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Pasta do projeto:", os.getcwd())

In [ ]:
# Confere se os arquivos dos notebooks anteriores chegaram
for arq in ["avaliacao/respostas_sem_contexto.jsonl", "avaliacao/respostas_rag.jsonl", "avaliacao/respostas_rag_lora.jsonl"]:
    assert os.path.exists(arq), f"Falta {arq}: rode antes o notebook 02 e o 03 até o fim (ele salva no Drive)."
print("ok")

In [ ]:
import pandas as pd
from src.base_conhecimento import carregar_trechos
from src.avaliacao import carregar_jsonl, corrigir, placar
from src.config import ARQ_TESTE
from src.experimento import CONFIGS, arq_respostas
trechos = carregar_trechos(); por_id = {t["id"]: t for t in trechos}
ids, fids = set(por_id), {t["fonte_id"] for t in trechos}
teste = carregar_jsonl(ARQ_TESTE); casos = {c["id"]: c for c in teste}
respostas = {n: {s["id"]: s for s in carregar_jsonl(arq_respostas(n))} for n in CONFIGS}
correcoes = {n: [corrigir(casos[i], respostas[n][i], ids, fids) for i in casos] for n in CONFIGS}
tab = pd.DataFrame([placar(correcoes[n], n) for n in CONFIGS]).set_index("configuracao")
tab.to_csv("avaliacao/placar.csv"); open("avaliacao/placar.md", "w").write(tab.to_markdown())
tab

In [ ]:
import matplotlib.pyplot as plt
cols = ["acuracia_%", "alucinacao_%", "recusa_correta_nao_consta_%", "json_valido_%", "citacao_correta_%"]
ax = tab[cols].T.plot.bar(figsize=(10, 4.5), rot=15)
ax.set_ylabel("%"); ax.set_title("Placar das três configurações (conjunto de teste)"); ax.set_ylim(0, 105)
for c in ax.containers: ax.bar_label(c, fmt="%.0f", fontsize=7)
plt.tight_layout(); plt.savefig("avaliacao/placar.png", dpi=120); plt.show()

## Por categoria de pergunta

In [ ]:
por_cat = pd.concat({n: pd.DataFrame(correcoes[n]).groupby("categoria")["acerto"].mean().mul(100).round(0)
                     for n in CONFIGS}, axis=1)
por_cat

## Demonstração de alucinações (sem contexto alucina, RAG acerta)

In [ ]:
linhas = []
for i, c in casos.items():
    a = {n: next(x for x in correcoes[n] if x["id"] == i) for n in CONFIGS}
    if a["sem_contexto"]["alucinacao"] and a["rag"]["acerto"]:
        linhas.append(f"### {i} – {c['pergunta']}\n- **Gabarito:** {c['resposta_gabarito']}\n"
                      f"- **Sem contexto (alucinou):** {respostas['sem_contexto'][i]['resposta']} "
                      f"(fontes: {respostas['sem_contexto'][i]['fontes']})\n"
                      f"- **RAG:** {respostas['rag'][i]['resposta']} (fontes: {respostas['rag'][i]['fontes']})\n")
open("avaliacao/alucinacoes.md", "w").write("# Alucinações demonstradas\n\n" + "\n".join(linhas))
print(len(linhas), "casos"); print("\n".join(linhas[:5]))

## Erros da melhor configuração (insumo para os 5 erros do relatório)

In [ ]:
melhor = tab["acuracia_%"].idxmax(); print("melhor configuração:", melhor)
saida = []
for x in correcoes[melhor]:
    if x["acerto"] and x["json_valido"] and x["citacao_correta"] is not False: continue
    c, r = casos[x["id"]], respostas[melhor][x["id"]]
    saida.append(f"### {x['id']} ({c['tipo']}, {c.get('categoria','')})\n- Pergunta: {c['pergunta']}\n"
                 f"- Gabarito: {c['resposta_gabarito']} | trechos {c['trechos_gabarito']}\n"
                 f"- Recuperados: {r['recuperados']} (score top1 {r['score_top1']})\n"
                 f"- Saída bruta: `{r['bruto'][:400]}`\n- Correção: {x}\n")
open("avaliacao/erros_candidatos.md", "w").write(f"# Erros de {melhor}\n\n" + "\n".join(saida))
print(len(saida), "erros"); print("\n".join(saida))

In [ ]:
# Salva os resultados no Google Drive (para o próximo notebook usar)
import shutil
FIXOS = {"casos_teste.jsonl", "lora_treino.jsonl", ".gitkeep"}
if "google.colab" in sys.modules:
    for pasta in ["avaliacao", "modelos"]:
        if os.path.exists(pasta):
            shutil.copytree(pasta, os.path.join(PERSIST, pasta), dirs_exist_ok=True,
                            ignore=lambda d, nomes: [n for n in nomes if n in FIXOS or n.endswith("_ckpt")])
    print("Salvo no Drive:", {p: sorted(os.listdir(os.path.join(PERSIST, p)))
                              for p in os.listdir(PERSIST) if os.path.isdir(os.path.join(PERSIST, p))})

## Baixar todos os resultados (para subir no GitHub)

In [ ]:
import subprocess
subprocess.run(["zip", "-qr", "/content/resultados.zip", "avaliacao", "modelos", "-x", "*_ckpt/*"], check=True)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download("/content/resultados.zip")